# Interstellar candidate exploration
Queries the ALeRCE TAP service for unlinked, trailed LSST detections — the raw candidate pool for hyperbolic orbit fitting.

In [ ]:
import pyvo
import requests
from requests.adapters import HTTPAdapter

class HTTPSRedirectAdapter(HTTPAdapter):
    """Force all HTTP requests to HTTPS — the ALeRCE TAP server returns HTTP
    job URLs in its async responses but only listens on HTTPS."""
    def send(self, request, **kwargs):
        request.url = request.url.replace('http://', 'https://')
        return super().send(request, **kwargs)

session = requests.Session()
session.mount('http://', HTTPSRedirectAdapter())

tap = pyvo.dal.TAPService('https://tap.alerce.online/tap', session=session)

In [7]:
from astropy.time import Time

def mjd_to_date(mjd):
    return Time(mjd, format='mjd').strftime('%Y-%m-%d')

def date_to_mjd(date):
    """date: 'YYYY-MM-DD'"""
    return float(Time(date, format='iso').mjd)

# quick test
print(mjd_to_date(61141.0))   # expect 2026-04-11
print(date_to_mjd('2026-04-11'))  # expect 61141.0

2026-04-11
61141.0


In [8]:
# lsst_detection has LSST-specific columns (trail fitting, ssobjectid, etc.)
# but no ra/dec/mjd — those are in the base detection table, joined on measurement_id
query = """
SELECT TOP 1000
    d.measurement_id, d.ra, d.dec, d.mjd, d.band,
    l.traillength, l.trailangle, l.trailchi2, l.trailflux, l.snr
FROM alerce_tap.detection AS d
JOIN alerce_tap.lsst_detection AS l ON d.measurement_id = l.measurement_id
WHERE l.ssobjectid IS NULL
  AND l.traillength > 0
ORDER BY d.mjd DESC
"""

results = tap.search(query)
df = results.to_table().to_pandas()
print(f"{len(df)} rows returned")
df.head()

1000 rows returned


,measurement_id,ra,dec,mjd,band,traillength,trailangle,trailchi2,trailflux,snr
0,170419912074330131,150.307248,3.387897,61178.986326,6,1.253264,28.792852,NaN,3042.884277,10.665019
1,170419912089534505,149.909375,1.504367,61178.986326,6,0.740237,108.669968,NaN,-3050.567627,14.601042
2,170419912090058763,150.051834,1.682753,61178.986326,6,1.013699,-5.357120,NaN,5082.186523,17.299822
3,170419912074330179,150.268517,3.317500,61178.986326,6,0.801751,8.494847,NaN,-6153.669922,18.063215
4,170419912087437338,149.734255,1.471337,61178.986326,6,0.738802,-0.279898,NaN,3452.843018,10.882829


In [4]:
# Check what nights are actually in the DB
result = tap.search("""
    SELECT MIN(d.mjd) AS mjd_min, MAX(d.mjd) AS mjd_max, COUNT(*) AS n
    FROM alerce_tap.detection AS d
    JOIN alerce_tap.lsst_detection AS l ON d.measurement_id = l.measurement_id
    WHERE l.ssobjectid IS NULL
""")
result.to_table().to_pandas()

,mjd_min,mjd_max,n
0,60984.253304,61178.986326,1145646


In [9]:
# Find which nights actually have data
result_nights = tap.search("""
    SELECT FLOOR(d.mjd) AS night, COUNT(*) AS n_detections
    FROM alerce_tap.detection AS d
    JOIN alerce_tap.lsst_detection AS l ON d.measurement_id = l.measurement_id
    WHERE l.ssobjectid IS NULL
    GROUP BY FLOOR(d.mjd)
    ORDER BY night
""")
df_nights = result_nights.to_table().to_pandas()
print(df_nights.to_string())


      night  n_detections
0   60984.0             2
1   60985.0             4
2   60987.0             6
3   60988.0             2
4   60990.0             4
5   60991.0             8
6   60998.0             1
7   61002.0            10
8   61003.0            44
9   61004.0             2
10  61006.0            74
11  61007.0            15
12  61010.0             2
13  61019.0            12
14  61020.0             4
15  61024.0           492
16  61026.0            81
17  61028.0           333
18  61029.0           372
19  61030.0            28
20  61032.0           424
21  61033.0           464
22  61034.0            62
23  61040.0            12
24  61041.0           102
25  61042.0            57
26  61043.0            53
27  61044.0            33
28  61045.0            24
29  61046.0           166
30  61048.0           194
31  61049.0           276
32  61050.0           609
33  61051.0           979
34  61052.0            87
35  61053.0           940
36  61054.0           263
37  61056.0 

In [13]:
import pandas as pd
import psycopg2
import requests
import time
from io import StringIO, BytesIO
from astropy.io.votable import parse as parse_votable
from datetime import datetime

def fetch_async(label, query, poll_interval=15):
    t0 = time.time()
    print(f"[{datetime.now():%H:%M:%S}] Submitting {label}...", flush=True)
    job = tap.run_async(query)
    job_url = job.url.replace('http://', 'https://')
    print(f"  job URL: {job_url}", flush=True)

    while True:
        try:
            phase = requests.get(f"{job_url}/phase", timeout=30).text.strip()
        except Exception as e:
            print(f"  [{elapsed(t0)}] poll error ({e}), retrying...", flush=True)
            time.sleep(poll_interval)
            continue
        print(f"  [{elapsed(t0)}] phase: {phase}", flush=True)
        if phase == 'COMPLETED':
            break
        if phase in ('ERROR', 'ABORTED'):
            raise RuntimeError(f"Job failed: {phase}")
        time.sleep(poll_interval)

    print(f"  [{elapsed(t0)}] Downloading result...", flush=True)
    r = requests.get(f"{job_url}/results/result", timeout=300)
    vot = parse_votable(BytesIO(r.content))
    df = vot.get_first_table().to_table().to_pandas()
    print(f"  [{elapsed(t0)}] {len(df):,} rows fetched", flush=True)
    return df

def elapsed(t0):
    s = int(time.time() - t0)
    return f"{s//60}m{s%60:02d}s"

def pg_copy(cur, df, table):
    print(f"  Inserting {len(df):,} rows into {table}...", flush=True)
    buf = StringIO()
    df.to_csv(buf, index=False, header=False, na_rep='')
    buf.seek(0)
    cur.copy_from(buf, table, sep=',', null='', columns=df.columns.tolist())
    print(f"  Insert complete.", flush=True)

con = psycopg2.connect(dbname='interstellar')
con.autocommit = True
cur = con.cursor()

cur.execute("SELECT COUNT(*) FROM lsst_detection")
if cur.fetchone()[0] == 0:
    df = fetch_async('lsst_detection', """
        SELECT measurement_id, ssobjectid, snr,
               psfflux, psffluxerr, traillength, trailangle,
               pixelflags_bad, pixelflags_saturated
        FROM alerce_tap.lsst_detection
        WHERE ssobjectid IS NULL
    """)
    pg_copy(cur, df, 'lsst_detection')
else:
    cur.execute("SELECT COUNT(*) FROM lsst_detection")
    print(f"lsst_detection already loaded ({cur.fetchone()[0]:,} rows), skipping")

cur.execute("SELECT COUNT(*) FROM detection")
if cur.fetchone()[0] == 0:
    df = fetch_async('detection', """
        SELECT measurement_id, ra, dec, mjd, band
        FROM alerce_tap.detection
        WHERE sid IN (1, 2)
    """)
    pg_copy(cur, df, 'detection')
else:
    cur.execute("SELECT COUNT(*) FROM detection")
    print(f"detection already loaded ({cur.fetchone()[0]:,} rows), skipping")

cur.close()
con.close()
print(f"\n[{datetime.now():%H:%M:%S}] Done. Query locally with: psql -d interstellar")

[03:00:34] Submitting lsst_detection...


ConnectTimeout: HTTPConnectionPool(host='tap.alerce.online', port=80): Max retries exceeded with url: /__system__/tap/run/async/s_7edcqb (Caused by ConnectTimeoutError(<HTTPConnection(host='tap.alerce.online', port=80) at 0xffff5aee2030>, 'Connection to tap.alerce.online timed out. (connect timeout=None)'))

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

# Pairing parameters
MIN_DT_HR     = 0.1      # min 6 min apart (avoid same-visit duplicates)
MAX_DT_HR     = 3.0      # max 3 hours apart (same night)
MIN_SEP_ARCSEC = 2.0     # must have moved > 2" (above PSF noise floor)
MAX_SEP_ARCSEC = 3600.0  # max 1 deg/hr at 1 hr — generous upper bound

coords = SkyCoord(ra=night.ra.values * u.deg, dec=night.dec.values * u.deg)
mjds   = night.mjd.values

# search_around_sky returns all pairs within the spatial cap; we filter by time after
idx1, idx2, ang_seps, _ = coords.search_around_sky(coords, MAX_SEP_ARCSEC * u.arcsec)

pairs = []
for i, j, sep in zip(idx1, idx2, ang_seps):
    if j <= i:           # avoid self-matches and duplicates
        continue
    dt_hr = (mjds[j] - mjds[i]) * 24
    if not (MIN_DT_HR < dt_hr < MAX_DT_HR):
        continue
    sep_arcsec = sep.arcsec
    if sep_arcsec < MIN_SEP_ARCSEC:
        continue
    pairs.append({
        'id1':             night.measurement_id.iloc[i],
        'id2':             night.measurement_id.iloc[j],
        'ra1':             night.ra.iloc[i],  'dec1': night.dec.iloc[i],
        'ra2':             night.ra.iloc[j],  'dec2': night.dec.iloc[j],
        'mjd1':            mjds[i],           'mjd2': mjds[j],
        'dt_hr':           dt_hr,
        'sep_arcsec':      sep_arcsec,
        'rate_arcsec_hr':  sep_arcsec / dt_hr,
        'pa_deg':          coords[i].position_angle(coords[j]).deg,
    })

df_pairs = pd.DataFrame(pairs)
print(f"{len(df_pairs)} candidate intra-night pairs")
df_pairs.sort_values('rate_arcsec_hr', ascending=False).head(20)